# List and Show

Demonstrating how to list samples and graphs and retrieve sequences using the Python API.
See `list-and-show.md` for the equivalent CLI workflow.

In [ ]:
import os
import tempfile

import gen

tmpdir = tempfile.mkdtemp()
repo = gen.Repository(os.path.join(tmpdir, "gen"))

In [ ]:
fasta_path = os.path.join(tmpdir, "simple.fa")
with open(fasta_path, "w") as f:
    f.write(">m123\nATCGATCGATCGATCGATCGGGAACACACAGAGA\n")

vcf_path = os.path.join(tmpdir, "simple.vcf")
with open(vcf_path, "w") as f:
    f.write(
        "##fileformat=VCFv4.1\n"
        "##contig=<ID=m123,length=34>\n"
        '##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">\n'
        "#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tunknown\tG1\tfoo\n"
        "m123\t3\t.\tCG\tC\t.\t.\t.\tGT\t1/1\t1/1\t1/1\n"
        "m123\t10\t.\tT\tTAGA\t.\t.\t.\tGT\t1/1\t0/0\t0/0\n"
    )

repo.import_reference_fasta(fasta_path, "reference")
# Apply the full VCF at once — all samples processed together
repo.update_with_vcf(vcf_path, reference="reference")
print("Import and update complete.")


## List samples

Equivalent to `gen list-samples`. No dedicated Python method exists;
derive the sample list from `get_block_groups()`.

In [ ]:
bgs = repo.get_block_groups()
samples = sorted(set(bg.sample_name for bg in bgs))
print("Samples:")
for s in samples:
    print(f"  {s}")

## List graphs for a sample

Equivalent to `gen list-graphs --sample foo`. No dedicated Python method exists;
filter `get_block_groups()` by sample name.

In [ ]:
foo_graphs = [bg.name for bg in bgs if bg.sample_name == "foo"]
print("Graphs for sample 'foo':")
for g in foo_graphs:
    print(f"  {g}")

## Get a sequence

Equivalent to `gen get-sequence --sample foo --graph m123`.

`get-sequence` and its `--start`/`--end` variant have no direct Python API equivalent yet.
The closest approach is to export to FASTA and read it back.

In [ ]:
fasta_out = os.path.join(tmpdir, "foo.fa")
repo.export_fasta(fasta_out, sample="foo")

with open(fasta_out) as f:
    print(f.read())